# 05 — Retrieval evaluation and hybrid search

Stream C, tasks **C3–C5**. This notebook compares dense encoders on the LEDGAR test split, then evaluates BM25 and reciprocal-rank fusion (RRF). The automatic benchmark defines relevance as **same LEDGAR label** and uses the 60,000-clause training split as the retrieval bank. The lawyer-question benchmark is separate and is scored only after its relevant training-clause indices have been judged by a human.

The encoder comparison can take time and download three models. It does not require a GPU, though an available accelerator can speed up embedding. All index embeddings are cached under the gitignored `data/indexes/` directory.

## 1. Load the fixed train bank and test queries

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import RESULTS_DIR, ROOT
from src.data import load_ledgar, to_pandas
from src.retrieval.dense import MODEL_NAME, build_dense_index
from src.retrieval.evaluation import evaluate_manual_judgments, evaluate_same_label_with_bank
from src.retrieval.hybrid import BM25Retriever, HybridRetriever
from src.utils import set_seed

set_seed(42)
dataset = load_ledgar()
train_df = to_pandas(dataset, "train")
test_df = to_pandas(dataset, "test")
bank_labels = dict(zip(train_df["idx"].astype(int), train_df["label"].astype(int)))

print(f"Bank: {len(train_df):,} training clauses | Test queries: {len(test_df):,}")

## 2. Compare three sentence-transformer encoders (C4)

For each test clause, retrieve the top 10 training clauses. A result is relevant iff its label equals the test clause's label. P@5 divides by five; MRR is computed over the returned top-10; nDCG uses binary relevance at 10. This is a label-retrieval benchmark, not a judgment that the retrieved text answers every semantic detail.

In [ ]:
encoder_names = [
    MODEL_NAME,
    "sentence-transformers/all-mpnet-base-v2",
    "BAAI/bge-small-en-v1.5",
]

dense_retrievers = {}
automatic_rows = []
for model_name in encoder_names:
    print(f"\nBuilding/loading index: {model_name}")
    retriever = build_dense_index(train_df, model_name=model_name, batch_size=64)
    dense_retrievers[model_name] = retriever
    started = time.perf_counter()
    rankings = retriever.search_many(test_df["text"].tolist(), k=10)
    elapsed_s = time.perf_counter() - started
    metrics = evaluate_same_label_with_bank(
        test_df["label"].tolist(), rankings, bank_labels, precision_k=5, ndcg_k=10
    )
    automatic_rows.append({
        "method": model_name,
        "query_set": "LEDGAR test; relevance=same label",
        **metrics,
        "search_time_s": round(elapsed_s, 2),
        "index_cache_hit": retriever.cache_hit,
    })
    del rankings

dense_results_df = pd.DataFrame(automatic_rows)
display(dense_results_df)

## 3. Evaluate BM25 and dense + BM25 RRF (C5)

The lexical baseline is BM25 over the same training bank. Hybrid search fuses dense and BM25 ranks using RRF with constant 60. The optional `predicted_label` argument filters results to a classifier-predicted class; this notebook evaluates the unfiltered form so retrieval quality is not conflated with classifier errors.

In [ ]:
mini_retriever = dense_retrievers[MODEL_NAME]
bm25_retriever = BM25Retriever(train_df)
hybrid_retriever = HybridRetriever(mini_retriever, bm25_retriever)

test_queries = test_df["text"].tolist()
for method_name, retrieve_many in [
    ("BM25", lambda: bm25_retriever.search_many(test_queries, k=10)),
    ("MiniLM + BM25 (RRF, c=60)", lambda: hybrid_retriever.search_many(test_queries, k=10)),
]:
    print(f"Evaluating {method_name} on all test queries")
    started = time.perf_counter()
    rankings = retrieve_many()
    elapsed_s = time.perf_counter() - started
    metrics = evaluate_same_label_with_bank(
        test_df["label"].tolist(), rankings, bank_labels, precision_k=5, ndcg_k=10
    )
    automatic_rows.append({
        "method": method_name,
        "query_set": "LEDGAR test; relevance=same label",
        **metrics,
        "search_time_s": round(elapsed_s, 2),
        "index_cache_hit": None,
    })
    del rankings

automatic_results_df = pd.DataFrame(automatic_rows)
display(automatic_results_df)

## 4. Review and score the lawyer questions (C3)

Each query in `eval/retrieval_queries.jsonl` starts with `judgment_status: pending_manual_review`. Inspect the top 10 candidates from dense MiniLM, BM25 and hybrid together (the pooled results form the judgment set). Add the training-bank `idx` values that genuinely answer the query to `relevant_train_indices`, then change the status to `reviewed`. Do not mark a result relevant merely because it shares a broad topic or LEDGAR label; numeric and jurisdiction constraints must match. An empty list is a valid judgment only after confirming that none of the inspected candidates answers the question.

In [ ]:
query_path = ROOT / "eval" / "retrieval_queries.jsonl"
lawyer_queries = [json.loads(line) for line in query_path.read_text().splitlines() if line.strip()]
pending = [query for query in lawyer_queries if query["judgment_status"] != "reviewed"]
print(f"Lawyer queries: {len(lawyer_queries)} | pending human review: {len(pending)}")

lawyer_query_texts = [query["query"] for query in lawyer_queries]
candidate_rankings = {
"MiniLM": mini_retriever.search_many(lawyer_query_texts, k=10),
"BM25": bm25_retriever.search_many(lawyer_query_texts, k=10),
"Hybrid RRF": hybrid_retriever.search_many(lawyer_query_texts, k=10),
}
candidate_rows = []
for query_position, query in enumerate(lawyer_queries):
    for method_name, rankings in candidate_rankings.items():
        for rank, result in enumerate(rankings[query_position], start=1):
            candidate_rows.append({
                "query_id": query["id"],
                "query": query["query"],
                "method": method_name,
                "rank": rank,
                "idx": result["idx"],
                "label_name": result["label_name"],
                "score": round(result["score"], 4),
                "text": result["text"],
            })

candidate_df = pd.DataFrame(candidate_rows)
display(candidate_df)

In [ ]:
reviewed = [query for query in lawyer_queries if query["judgment_status"] == "reviewed"]
if not reviewed:
    print("No manually judged queries yet; manual-query metrics are intentionally not reported.")
else:
    reviewed_texts = [query["query"] for query in reviewed]
    reviewed_rankings = {
        "MiniLM": mini_retriever.search_many(reviewed_texts, k=10),
        "BM25": bm25_retriever.search_many(reviewed_texts, k=10),
        "Hybrid RRF": hybrid_retriever.search_many(reviewed_texts, k=10),
    }
    judgments = {
        query["id"]: set(map(int, query["relevant_train_indices"]))
        for query in reviewed
    }
    manual_rows = []
    for method_name, rankings in reviewed_rankings.items():
        query_rankings = {
            query["id"]: ranking
            for query, ranking in zip(reviewed, rankings)
        }
        manual_rows.append({
            "method": method_name,
            **evaluate_manual_judgments(query_rankings, judgments),
        })
    display(pd.DataFrame(manual_rows))

## 5. Save the automatic comparison

This writes the versioned summary table. The held-out test split is used here only for the pre-specified retrieval benchmark; changing model or index decisions based on these results should be documented.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary_path = RESULTS_DIR / "retrieval_summary.csv"
automatic_results_df.to_csv(summary_path, index=False)
print(f"Saved {len(automatic_results_df)} retrieval rows to {summary_path}")